In [ ]:
import segmentation_models as sm
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam
from wandb.keras import WandbCallback

from networks import *

print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
wandb.login()

In [ ]:
sweep_config = {
    'method': 'grid',  #grid, random
    'metric': {
        'name': 'val_dice_coef',
        'goal': 'maximize',
    },
    'parameters': {
        'learning_rate': {
            'values': [1e-2, 1e-3, 1e-4]
        },
        'momentum': {
            'values': [0.5, 0.9]
        },
        'batch_size': {
            'values': [2, 4, 8, 16]
        },
        'dropout_rate': {
            'values': [0.0, 0.25, 0.5]
        },
        'num_filters': {
            'values': [8, 16, 32, 64]
        },
    }
}

In [ ]:
def train():
    # Initialize wandb run
    run = wandb.init()

    # Retrieve hyperparameters from the sweep configuration
    config = run.config
    learning_rate = config.learning_rate
    momentum = config.momentum
    batch_size = config.batch_size
    dropout_rate = config.dropout_rate
    num_filters = config.num_filters

    # Constant parameters
    name = 'unet'
    height = 192
    width = 256
    channels = 3
    input_shape = (height, width, channels)
    num_classes = 8
    batchnorm = True
    final_activation = 'softmax'
    deep_supervision = False
    epochs = 100
    steps_per_epoch = 500

    # Create the model
    model = create_model(
        name=name, input_shape=input_shape, num_classes=num_classes, num_filters=num_filters,
        dropout=dropout_rate, batchnorm=batchnorm, final_activation=final_activation, deep_supervision=deep_supervision,
    )

    # Load and preprocess your dataset
    dataset_path = 'input/'
    train_ds, val_ds, test_ds = load_dataset(
        dataset_path, val_split=0.1, batch_size=batch_size,
        height=height, width=width, channels=channels,
        preprocess=None, augment=True, repeat=True,
    )

    # Compile the model
    loss = sm.losses.categorical_focal_dice_loss
    iou = sm.metrics.IOUScore(threshold=0.5)
    f1 = sm.metrics.FScore(threshold=0.5)
    model.compile(
        optimizer=Adam(learning_rate=learning_rate, beta_1=momentum),
        loss=loss,
        metrics=['accuracy', jaccard_coef, dice_coef, iou, f1],
    )

    # Train the model
    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3, verbose=1),
        EarlyStopping(monitor='val_loss', patience=5, verbose=1),
        VisualizeProgress(
            test_data=next(iter(val_ds)), every_n_epochs=1, num_samples=5,
            convert_mask=one_hot_to_rgb, deep_supervision=deep_supervision, log=True,
        ),
        WandbCallback(),
    ]

    history = model.fit(
        train_ds, validation_data=val_ds, validation_steps=len(val_ds),
        epochs=epochs, steps_per_epoch=steps_per_epoch,
        callbacks=callbacks, verbose=1,
    )
    plot_history(history)

    # Evaluate the model
    test_loss, test_acc, test_jaccard, test_dice, test_iou, test_f1 = model.evaluate(test_ds, verbose=1)
    wandb.log({
        'test_loss': test_loss,
        'test_accuracy': test_acc,
        'test_jaccard': test_jaccard,
        'test_dice': test_dice,
        'test_iou': test_iou,
        'test_f1': test_f1,
    })


def create_model(name, input_shape, num_classes, num_filters, dropout, batchnorm, final_activation, deep_supervision):
    name = name.lower()
    if name == 'unet':
        return UNet(
            input_shape=input_shape,
            num_classes=num_classes,
            num_filters=num_filters,
            dropout=dropout,
            batchnorm=batchnorm,
            final_activation=final_activation,
        )
    elif name == 'unet++':
        return UNetPlusPlus(
            input_shape=input_shape,
            num_classes=num_classes,
            num_filters=num_filters,
            dropout=dropout,
            batchnorm=batchnorm,
            final_activation=final_activation,
            deep_supervision=deep_supervision,
        )
    elif name == 'unet3+':
        return UNet3Plus(
            input_shape=input_shape,
            num_classes=num_classes,
            num_filters=num_filters,
            dropout=dropout,
            batchnorm=batchnorm,
            final_activation=final_activation,
            deep_supervision=deep_supervision,
        )
    elif name == 'attention_unet':
        return Attention_UNet(
            input_shape=input_shape,
            num_classes=num_classes,
            num_filters=num_filters,
            dropout=dropout,
            batchnorm=batchnorm,
            final_activation=final_activation,
        )
    elif name == 'attention_resunet':
        return Attention_ResUNet(
            input_shape=input_shape,
            num_classes=num_classes,
            num_filters=num_filters,
            dropout=dropout,
            batchnorm=batchnorm,
            final_activation=final_activation,
        )
    elif name == 'suim-net':
        return SuimNet(
            input_shape=input_shape,
            num_classes=num_classes,
        )
    raise ValueError(f'Unknown model name: {name}')


In [ ]:
# Create a sweep with the given configuration
sweep_id = wandb.sweep(sweep_config, project='PVID-assignment-4')

# Run the sweep using the train function
wandb.agent(sweep_id, function=train)